# 1. Install Required Libraries

In [4]:
# ============================================================
# 01. Import libraries
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    XLMRobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


# 2. Configurations

For reproducability, let's set randome seeds.

In [5]:
# ============================================================
# 02. Set random seeds
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)

Random seed: 42


In [6]:
# ============================================================
# 03. Experiment configuration
# ============================================================

MODEL_NAME = "FacebookAI/xlm-roberta-base"

# Change these according to your dataset.
TEXT_COLUMN = "text"
LABEL_COLUMN = "label"

# Number of classification classes.
NUM_LABELS = 2

MAX_LENGTH = 128

OUTPUT_DIR = "/kaggle/working/xlm_r_frozen_baseline"

print("Model:", MODEL_NAME)
print("Max sequence length:", MAX_LENGTH)
print("Number of labels:", NUM_LABELS)

Model: FacebookAI/xlm-roberta-base
Max sequence length: 128
Number of labels: 2


# 3. Dataset

In [7]:
# ============================================================
# 04. Load JSONL dataset
# ============================================================

TRAIN_PATH = "/kaggle/input/datasets/uom220676x/clean-sinhala-train-and-test/clean_sinhala_train.jsonl"
TEST_PATH = "/kaggle/input/datasets/uom220676x/clean-sinhala-train-and-test/clean_sinhala_test.jsonl"

train_df = pd.read_json(TRAIN_PATH, lines=True)
test_df = pd.read_json(TEST_PATH, lines=True)

# ------------------------------------------------------------
# Split the ORIGINAL TRAINING SET into:
#
#   90% -> training
#   10% -> validation
#
# The existing TEST SET is NOT touched.
# ------------------------------------------------------------
train_df, valid_df = train_test_split(
    train_df,
    test_size=0.10,
    random_state=SEED,
    stratify=train_df[LABEL_COLUMN]
)

# Reset dataframe indices after splitting.
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\nAfter splitting:")
print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

# Check class distributions.
print("\nTrain label distribution:")
print(train_df[LABEL_COLUMN].value_counts())

print("\nValidation label distribution:")
print(valid_df[LABEL_COLUMN].value_counts())

print("\nTest label distribution:")
print(test_df[LABEL_COLUMN].value_counts())

display(train_df.head())


After splitting:
Train: (6750, 6)
Validation: (750, 6)
Test: (2500, 6)

Train label distribution:
label
0    3892
1    2858
Name: count, dtype: int64

Validation label distribution:
label
0    432
1    318
Name: count, dtype: int64

Test label distribution:
label
0    1485
1    1015
Name: count, dtype: int64


,post_id,text,tokens,rationales,label,split
0,798491968795975680,@USER @USER පිංගොනා අහනෝ තෝ වැස්සි ඔන්ලයින් ඩ...,@USER @USER පිංගොනා අහනෝ තෝ වැස්සි ඔන්ලයින් ඩ ...,"[0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0]",1,train
1,1216671615318216704,@USER අනේ උඹේ අම්මට හුකාපං පොන්න වේසිගෙ පුතා. ...,@USER අනේ උඹේ අම්මට හුකාපං පොන්න වේසිගෙ පුතා ....,"[0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, ...",1,train
2,865966167843684352,@USER තෝ කොළඹ යනවා කිව්වා නේද හුත්තෝ,@USER තෝ කොළඹ යනවා කිව්වා නේද හුත්තෝ,"[0, 1, 0, 0, 0, 0, 1]",1,train
3,793486893417725952,@USER @USER තිබ්බත් සල්ලි වියදම් කරන්ඩ එපා...,@USER @USER තිබ්බත් සල්ලි වියදම් කරන්ඩ එපා. බො...,[],0,train
4,1447800275356454912,"රටේ දරුවො ගැන නොහිතන කාලකණ්ණි පාලකයෝ., පටු වාස...","රටේ දරුවො ගැන නොහිතන කාලකණ්ණි පාලකයෝ . , පටු ව...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,train


In [8]:
# ============================================================
# 05. Inspect dataset
# ============================================================

print("Columns:")
print(train_df.columns.tolist())

print("\nFirst few samples:")
display(train_df.head())

print("\nLabel distribution:")
print(train_df[LABEL_COLUMN].value_counts())

print("\nLabel distribution (%):")
print(train_df[LABEL_COLUMN].value_counts(normalize=True) * 100)

Columns:
['post_id', 'text', 'tokens', 'rationales', 'label', 'split']

First few samples:


,post_id,text,tokens,rationales,label,split
0,798491968795975680,@USER @USER පිංගොනා අහනෝ තෝ වැස්සි ඔන්ලයින් ඩ...,@USER @USER පිංගොනා අහනෝ තෝ වැස්සි ඔන්ලයින් ඩ ...,"[0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0]",1,train
1,1216671615318216704,@USER අනේ උඹේ අම්මට හුකාපං පොන්න වේසිගෙ පුතා. ...,@USER අනේ උඹේ අම්මට හුකාපං පොන්න වේසිගෙ පුතා ....,"[0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, ...",1,train
2,865966167843684352,@USER තෝ කොළඹ යනවා කිව්වා නේද හුත්තෝ,@USER තෝ කොළඹ යනවා කිව්වා නේද හුත්තෝ,"[0, 1, 0, 0, 0, 0, 1]",1,train
3,793486893417725952,@USER @USER තිබ්බත් සල්ලි වියදම් කරන්ඩ එපා...,@USER @USER තිබ්බත් සල්ලි වියදම් කරන්ඩ එපා. බො...,[],0,train
4,1447800275356454912,"රටේ දරුවො ගැන නොහිතන කාලකණ්ණි පාලකයෝ., පටු වාස...","රටේ දරුවො ගැන නොහිතන කාලකණ්ණි පාලකයෝ . , පටු ව...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,train



Label distribution:
label
0    3892
1    2858
Name: count, dtype: int64

Label distribution (%):
label
0    57.659259
1    42.340741
Name: proportion, dtype: float64


In [9]:
# ============================================================
# 07. Keep only text and label columns
# ============================================================

train_df = train_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()
valid_df = valid_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()
test_df = test_df[[TEXT_COLUMN, LABEL_COLUMN]].copy()

# Remove missing values.
train_df = train_df.dropna()
valid_df = valid_df.dropna()
test_df = test_df.dropna()

# Make sure text is a string.
train_df[TEXT_COLUMN] = train_df[TEXT_COLUMN].astype(str)
valid_df[TEXT_COLUMN] = valid_df[TEXT_COLUMN].astype(str)
test_df[TEXT_COLUMN] = test_df[TEXT_COLUMN].astype(str)

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

Train: (6750, 2)
Validation: (750, 2)
Test: (2500, 2)


In [10]:
# ============================================================
# 08. Encode labels
# ============================================================

# Convert NumPy integer labels to normal Python integers.
train_df[LABEL_COLUMN] = train_df[LABEL_COLUMN].astype(int)
valid_df[LABEL_COLUMN] = valid_df[LABEL_COLUMN].astype(int)
test_df[LABEL_COLUMN] = test_df[LABEL_COLUMN].astype(int)

# Define the mapping explicitly.
label2id = {
    "non-hate": 0,
    "hate": 1,
}

id2label = {
    0: "non-hate",
    1: "hate",
}

# Create the numeric "labels" column used by the model.
train_df["labels"] = train_df[LABEL_COLUMN]
valid_df["labels"] = valid_df[LABEL_COLUMN]
test_df["labels"] = test_df[LABEL_COLUMN]

print("label2id:", label2id)
print("id2label:", id2label)

label2id: {'non-hate': 0, 'hate': 1}
id2label: {0: 'non-hate', 1: 'hate'}


In [11]:
# ============================================================
# 09. Convert pandas DataFrames to Hugging Face Dataset
# ============================================================

train_dataset = Dataset.from_pandas(
    train_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)

valid_dataset = Dataset.from_pandas(
    valid_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)

dataset = DatasetDict({
    "train": train_dataset,
    "validation": valid_dataset,
    "test": test_dataset,
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 6750
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 750
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 2500
    })
})


# 4. Processing

In [12]:
# ============================================================
# 10. Load original XLM-R tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded.")
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded.
Vocabulary size: 250002


In [13]:
# ============================================================
# 11. Inspect tokenization
# ============================================================

sample_texts = train_df[TEXT_COLUMN].head(5).tolist()

for text in sample_texts:

    tokens = tokenizer.tokenize(text)

    print("=" * 80)
    print("TEXT:")
    print(text)

    print("\nTOKENS:")
    print(tokens)

    print("\nNUMBER OF TOKENS:")
    print(len(tokens))

TEXT:
@USER  @USER පිංගොනා අහනෝ තෝ වැස්සි ඔන්ලයින් ඩ කියල යාලුවට කියන්න..

TOKENS:
['▁@', 'U', 'SER', '▁@', 'U', 'SER', '▁පි', 'ංග', 'ො', 'නා', '▁අහ', 'නෝ', '▁', 'තෝ', '▁වැ', 'ස්', 'සි', '▁ඔ', 'න්', 'ල', 'යින්', '▁ඩ', '▁කියල', '▁යාලුව', 'ට', '▁කියන්න', '.', '.']

NUMBER OF TOKENS:
28
TEXT:
@USER අනේ උඹේ අම්මට හුකාපං පොන්න වේසිගෙ පුතා. යස්මින් සූකා කියන්නෙ කව්ද කියලා තෝ අපිට කියන්න එපා පකෝ. ගොන් කැරියො තොපි මොන මඩ ප්‍රචාර ඇරියත් තොපිට ආයෙත් මේ රට කාබාසිනියා කරන්න දෙන්නෙ නෑ පකෝ. සක්කිලි වහල් වේසිගේ පුතා

TOKENS:
['▁@', 'U', 'SER', '▁අනේ', '▁උඹ', 'ේ', '▁අම්ම', 'ට', '▁හු', 'කා', 'පං', '▁පො', 'න්න', '▁වේ', 'සි', 'ගෙ', '▁පුතා', '.', '▁ය', 'ස්', 'මින්', '▁සූ', 'කා', '▁කියන්නෙ', '▁ක', 'ව්', 'ද', '▁කියලා', '▁', 'තෝ', '▁අපිට', '▁කියන්න', '▁එපා', '▁ප', 'කෝ', '.', '▁ගො', 'න්', '▁කැ', 'රිය', 'ො', '▁', 'තො', 'පි', '▁මොන', '▁මඩ', '▁ප්', '\u200d', 'ර', 'චාර', '▁ඇ', 'රිය', 'ත්', '▁', 'තො', 'පිට', '▁ආයෙත්', '▁මේ', '▁රට', '▁කා', 'බා', 'සි', 'නියා', '▁කරන්න', '▁දෙන්න', 'ෙ', '▁නෑ', '▁ප', 'කෝ', '.', '▁ස', '

In [14]:
# ============================================================
# 12. Tokenize dataset
# ============================================================

def tokenize_function(examples):
    """
    Tokenize the input text using the original XLM-R tokenizer.

    truncation=True:
        Truncate examples longer than MAX_LENGTH.

    max_length=MAX_LENGTH:
        Maximum sequence length.

    We intentionally do not perform additional text cleaning here.
    """

    return tokenizer(
        examples[TEXT_COLUMN],
        truncation=True,
        max_length=MAX_LENGTH,
    )


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
)

print(tokenized_dataset)

Map:   0%|          | 0/6750 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 6750
    })
    validation: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 750
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 2500
    })
})


In [15]:
# ============================================================
# 13. Data collator
# ============================================================

# Instead of padding every example to 128 tokens during preprocessing, use dynamic padding.
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Data collator created.")

Data collator created.


In [16]:
# ============================================================
# 14. Load XLM-R sequence classification model
# ============================================================

model = XLMRobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

print("Model loaded.")

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded.


In [17]:
# ============================================================
# 15. FULL XLM-R FINE-TUNING
# ============================================================

# Do NOT freeze any parameters.
# Both the XLM-R encoder and classification head will be trained.

for param in model.parameters():
    param.requires_grad = True

# Verify
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.2f}%")

Total parameters:     278,045,186
Trainable parameters: 278,045,186
Trainable percentage: 100.00%


In [18]:
# ============================================================
# 16. Verify which parameters are trainable
# ============================================================

total_params = 0
trainable_params = 0

for name, param in model.named_parameters():

    total_params += param.numel()

    if param.requires_grad:
        trainable_params += param.numel()

        print("TRAINABLE:", name)


print("\n" + "=" * 60)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

print(
    f"Trainable percentage: "
    f"{100 * trainable_params / total_params:.4f}%"
)

TRAINABLE: classifier.dense.weight
TRAINABLE: classifier.dense.bias
TRAINABLE: classifier.out_proj.weight
TRAINABLE: classifier.out_proj.bias
TRAINABLE: roberta.embeddings.word_embeddings.weight
TRAINABLE: roberta.embeddings.token_type_embeddings.weight
TRAINABLE: roberta.embeddings.LayerNorm.weight
TRAINABLE: roberta.embeddings.LayerNorm.bias
TRAINABLE: roberta.embeddings.position_embeddings.weight
TRAINABLE: roberta.encoder.layer.0.attention.self.query.weight
TRAINABLE: roberta.encoder.layer.0.attention.self.query.bias
TRAINABLE: roberta.encoder.layer.0.attention.self.key.weight
TRAINABLE: roberta.encoder.layer.0.attention.self.key.bias
TRAINABLE: roberta.encoder.layer.0.attention.self.value.weight
TRAINABLE: roberta.encoder.layer.0.attention.self.value.bias
TRAINABLE: roberta.encoder.layer.0.attention.output.dense.weight
TRAINABLE: roberta.encoder.layer.0.attention.output.dense.bias
TRAINABLE: roberta.encoder.layer.0.attention.output.LayerNorm.weight
TRAINABLE: roberta.encoder.layer

In [19]:
# ============================================================
# 17. Evaluation metrics
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    # Convert logits into predicted class IDs.
    predictions = np.argmax(logits, axis=-1)

    # Accuracy.
    accuracy = accuracy_score(
        labels,
        predictions
    )

    # Macro precision, recall and F1.
    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    # Weighted F1.
    _, _, weighted_f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }

In [20]:
# ============================================================
# 18. Training configuration
# ============================================================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training.
    num_train_epochs=10,

    # Batch sizes.
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    # Since XLM-R is frozen, the classifier can tolerate
    # a higher learning rate than full-model fine-tuning.
    learning_rate=2e-5,

    # Regularization.
    weight_decay=0.01,

    # Evaluation.
    eval_strategy="epoch",

    # Save the model at each epoch.
    save_strategy="epoch",

    # Keep the best checkpoint.
    load_best_model_at_end=True,

    # Select the best model using Macro F1.
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    # Logging.
    logging_strategy="steps",
    logging_steps=50,

    # Use GPU if available.
    fp16=torch.cuda.is_available(),

    # Reproducibility.
    seed=SEED,

    # Reduce unnecessary checkpoint storage.
    save_total_limit=2,

    report_to="none",
)

In [21]:
# ============================================================
# 19. Create Trainer
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer created.")

Trainer created.


In [22]:
# ============================================================
# 20. Train Model
# ============================================================

train_result = trainer.train()

print("\nTraining completed.")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
1,1.006894,0.924413,0.818667,0.822685,0.804420,0.809792,0.816037
2,0.844373,0.768903,0.834667,0.833826,0.825777,0.828873,0.833659
3,0.700581,0.745836,0.842667,0.839282,0.845999,0.840856,0.843436
4,0.611660,0.744221,0.840000,0.836083,0.836631,0.836351,0.840065
5,0.478348,0.743001,0.853333,0.850238,0.849035,0.849611,0.853207
6,0.439969,0.843281,0.846667,0.842450,0.846152,0.843948,0.847079
7,0.315487,1.015079,0.841333,0.837540,0.843597,0.839269,0.842038
8,0.295751,1.054494,0.836000,0.835826,0.826520,0.829991,0.834849
9,0.309118,1.177376,0.840000,0.839420,0.831237,0.834393,0.839025
10,0.241593,1.325334,0.836000,0.833174,0.829839,0.831314,0.835588


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


... (13 outputs left out)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Training completed.


In [23]:
# ============================================================
# 21. Evaluate on validation set
# ============================================================

validation_results = trainer.evaluate(
    tokenized_dataset["validation"]
)

print("\nValidation results:")

for key, value in validation_results.items():
    print(f"{key}: {value}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Validation results:
eval_loss: 0.7444207072257996
eval_accuracy: 0.8533333333333334
eval_macro_precision: 0.850238123064337
eval_macro_recall: 0.8490347658979734
eval_macro_f1: 0.8496106282447646
eval_weighted_f1: 0.8532071399405006
eval_runtime: 3.7727
eval_samples_per_second: 198.795
eval_steps_per_second: 3.181
epoch: 10.0


# 5. Evaluation

In [24]:
# ============================================================
# 22. Final test evaluation
# ============================================================

test_results = trainer.evaluate(
    tokenized_dataset["test"]
)

print("\nTest results:")

for key, value in test_results.items():
    print(f"{key}: {value}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Test results:
eval_loss: 0.8809556365013123
eval_accuracy: 0.8308
eval_macro_precision: 0.8241113532061808
eval_macro_recall: 0.8268613889303544
eval_macro_f1: 0.8253659182639836
eval_weighted_f1: 0.8311573432434115
eval_runtime: 12.4833
eval_samples_per_second: 200.267
eval_steps_per_second: 3.204
epoch: 10.0


In [25]:
# ============================================================
# 23. Detailed classification report
# ============================================================

predictions = trainer.predict(
    tokenized_dataset["test"]
)

predicted_labels = np.argmax(
    predictions.predictions,
    axis=-1
)

true_labels = predictions.label_ids

print(
    classification_report(
        true_labels,
        predicted_labels,
        target_names=[
            str(id2label[i])
            for i in range(NUM_LABELS)
        ],
        digits=4,
        zero_division=0
    )
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


              precision    recall  f1-score   support

    non-hate     0.8647    0.8478    0.8562      1485
        hate     0.7835    0.8059    0.7946      1015

    accuracy                         0.8308      2500
   macro avg     0.8241    0.8269    0.8254      2500
weighted avg     0.8317    0.8308    0.8312      2500



In [26]:
# ============================================================
# 25: Confusion matrix
# ============================================================

cm = confusion_matrix(
    true_labels,
    predicted_labels
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[1259  226]
 [ 197  818]]


In [27]:
# ============================================================
# 26: Save trained model and tokenizer
# ============================================================

FINAL_MODEL_DIR = "/kaggle/working/xlm_r_frozen_baseline_final"

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("Model saved to:")
print(FINAL_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to:
/kaggle/working/xlm_r_frozen_baseline_final


In [28]:
# ============================================================
# 27: Save experiment results
# ============================================================

experiment_results = {
    "experiment": "Frozen XLM-R + Classification Head",
    "model": MODEL_NAME,
    "tokenizer": "Original XLM-R tokenizer",
    "encoder_trainable": False,
    "classification_head_trainable": True,
    "max_length": MAX_LENGTH,
    "seed": SEED,

    "accuracy": test_results.get("eval_accuracy"),
    "macro_precision": test_results.get("eval_macro_precision"),
    "macro_recall": test_results.get("eval_macro_recall"),
    "macro_f1": test_results.get("eval_macro_f1"),
    "weighted_f1": test_results.get("eval_weighted_f1"),

    "total_parameters": total_params,
    "trainable_parameters": trainable_params,
    "trainable_percentage": (
        100 * trainable_params / total_params
    ),
}

results_df = pd.DataFrame([experiment_results])

display(results_df)

results_df.to_csv(
    "/kaggle/working/frozen_xlm_r_baseline_results.csv",
    index=False
)

print("Results saved.")

,experiment,model,tokenizer,encoder_trainable,classification_head_trainable,max_length,seed,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,total_parameters,trainable_parameters,trainable_percentage
0,Frozen XLM-R + Classification Head,FacebookAI/xlm-roberta-base,Original XLM-R tokenizer,False,True,128,42,0.8308,0.824111,0.826861,0.825366,0.831157,278045186,278045186,100.0


Results saved.


In [29]:
# ============================================================
# 28: Test individual examples
# ============================================================

def predict_text(text):

    # Tokenize input text.
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )

    # Move inputs to the same device as the model.
    device = next(model.parameters()).device

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Disable gradient calculation because we are predicting.
    with torch.no_grad():

        outputs = model(**inputs)

    # Convert logits to probabilities.
    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )

    predicted_id = torch.argmax(
        probabilities,
        dim=-1
    ).item()

    return {
        "text": text,
        "prediction": id2label[predicted_id],
        "confidence": probabilities[0, predicted_id].item(),
    }


# Try a few samples from your test set.
for text in test_df[TEXT_COLUMN].head(10):

    result = predict_text(text)

    print("=" * 80)
    print(result)

{'text': 'තේ නෙවෙයි තෝ බීලා ඉන්නෙ ගිනි වතුර', 'prediction': 'hate', 'confidence': 0.7304007411003113}
{'text': '@USER තුනක් ඕනේ නෑ එකක් ගැහුවනම් ඇති ලංකාවට ආපු ගමන් ඉල්ලා අස් වෙනවා කියලා', 'prediction': 'non-hate', 'confidence': 0.9812491536140442}
{'text': '@USER ,  @USER  @USER ටහුඩු ඒකි තනියම ජීවත් වෙන්න වෙරදරන එක්කෙනෙක් වෙන්න ඇති', 'prediction': 'non-hate', 'confidence': 0.9662250280380249}
{'text': '@USER මර්විනුත් මුද්දරයක් වෙයිද දන්නේ නැ', 'prediction': 'non-hate', 'confidence': 0.9902440905570984}
{'text': '@USER ගොසිප් පීපල් නෙමේ හුත්තො. තෝ වගේ හොරෙක්ට මේකෙ ආයෙ දඩයං හොයන්න දෙන්නෑ. එච්චරයි', 'prediction': 'hate', 'confidence': 0.9968692660331726}
{'text': 'අපිට නම් පෙන්නන්න තියෙන්නේ මේ ගල තමා! සුදු නෑ රතුයි!   ගුඩ් මෝනින් ෆ්\u200dරෙන්ස්!', 'prediction': 'non-hate', 'confidence': 0.989997148513794}
{'text': 'කේන්තියෙන්ම මං ඔයාගේ තාත්තට මෝඩයා කියල බැන්නා. තාත්තට පුදුමයි .. එයා ඇහුව ඔයා කොහොමද මට එහෙම කතා කරේ කියල. මං කේන්තියෙන් පිපිරී පිපිරී තව හයියෙන් කෑගැහුවා. මොඩයා බූරුවා ගොනා